# ASAP8 VIP somatic voltage characterization — multi-mouse, depth-binned

Cohort-level electrophysiological characterization of ASAP8+ VIP interneurons during passive Detection of Change.

This version generalizes the original single-session notebook in three deliberate ways:

1. **Cohort selection is by session type** (for example `A0`, `A1`, or `B2`) and includes all matching mice unless `TARGET_MICE` is specified.
2. **Cells are grouped by cortical depth**, using 100 µm half-open bins (`0–100`, `100–200`, …), with each ROI inheriting the `dmd1_depth` or `dmd2_depth` value from its session asset metadata.
3. **Synchrony is strictly within-session.** Pairwise spike metrics are constructed separately inside each asset/session and only then pooled. No pair can contain neurons from different mice or different sessions.

The main figures are keyed to the electrophysiology placeholders in the lab-meeting deck: spike detection/rate/bursting, isolated-spike waveforms and feature relationships, burst/plateau phenotypes, and ROI synchrony/cross-correlograms.

### Statistical unit
For descriptive plots, each ROI is shown as a cell-level observation. Because cells are nested in animals/sessions, the notebook also overlays or summarizes **mouse/session-level medians**. Pairwise synchrony values are even more strongly non-independent because each cell participates in multiple pairs; pooled synchrony summaries therefore use **one median per session × depth-bin pair** rather than treating every pair as an independent replicate.

### Processed-trace timing
The full-session H5 is treated as canonical for the analyzed time axis. The voltage postprocessor clips overlong source acquisition epochs at their paired behavior endpoints, so the source summary's `trialGlobalNLines` can legitimately exceed the saved processed trace length. The notebook therefore uses `trial_lengths_samples`, `timebase_sec`, and `sample_epoch` from the processed H5 and uses the summary only for source/ROI metadata.


## 0. Imports and plotting style

The palette is depth-coded rather than DMD-coded: superficial bins are peach and progressively deeper bins move toward blue.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import json
import math
import re
import warnings

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm, to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import ndimage, signal
from IPython.display import display, HTML

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

display(HTML("<style>.container { width:100% !important; }</style>"))

NAVY = "#1D457F"
CHARCOAL = "#2D2926"
DARK_BLUE = "#3E5685"
BLUE = "#5D74A5"
BLUE_FILL = "#B0CBE7"
DARK_PEACH = "#A8554E"
PEACH = "#EBA07E"
PEACH_FILL = "#F2C5B4"
CREAM = "#FEF7C7"
TEAL = "#64A8A8"
GRAY = "#7A7A7A"
LIGHT_GRAY = "#D9D9D9"

DEPTH_CMAP = LinearSegmentedColormap.from_list(
    "superficial_to_deep",
    [PEACH, CREAM, BLUE],
)
SYNC_CMAP = LinearSegmentedColormap.from_list(
    "sync",
    ["#F7F7F5", BLUE_FILL, BLUE, NAVY],
)

ZORDER = {
    "span": 0,
    "grid": 1,
    "trace": 2,
    "summary": 3,
    "marker": 4,
    "annotation": 5,
}

plt.rcParams.update({
    "figure.dpi": 115,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.transparent": False,
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "normal",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.0,
    "axes.axisbelow": True,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


## 1. Cohort definition

Change `TARGET_SESSION_TYPES` to the session type you want to characterize. A one-element list (for example `['A0']`) is the intended default. Multiple session types are allowed for exploratory distributional comparisons, but this notebook does **not** imply longitudinal cell identity across days.


In [ ]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")

TARGET_SESSION_TYPES = ["A0"]
TARGET_MICE = [852835,863774]          # None = every mouse with a matching session
MAX_SESSIONS = None         # useful for a quick development run

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
TRACE_VARIANT = "dff_robust_f0_trial"

DEPTH_BIN_UM = 100
TRACE_EXAMPLE_WINDOW_SEC = (20.0, 23.0)
SYNCH_RASTER_WINDOW_SEC = (20.0, 25.0)
N_BURST_EXAMPLES_PER_ROI = 5
MAX_CCG_PANELS = 6

SAVE_FIGURES = True
SAVE_TABLES = True
SAVE_SESSION_EVENT_TABLES = True
FAIL_FAST = False

session_tag = "-".join(map(str, TARGET_SESSION_TYPES))
OUTPUT_ROOT = BASE_PATH / "analysis" / "asap8_somatic_ephys_cohort" / session_tag
FIG_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
SESSION_TABLE_DIR = TABLE_DIR / "sessions"
for directory in (OUTPUT_ROOT, FIG_DIR, TABLE_DIR, SESSION_TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def save_panel(fig, name):
    if not SAVE_FIGURES:
        return
    for ext in ("png", "pdf", "svg"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", facecolor="white")

print("Output:", OUTPUT_ROOT)


In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

session_kwargs = dict(
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
)
if TARGET_MICE is not None:
    session_kwargs["subject_ids"] = list(TARGET_MICE)

session_df = registry.sessions(**session_kwargs).copy()

if "session_type" not in session_df.columns:
    raise KeyError(
        "Registry table has no 'session_type' column. "
        "Inspect session_df.columns and update the filter below."
    )

cohort_df = session_df.loc[
    session_df["session_type"].astype(str).isin(
        [str(x) for x in TARGET_SESSION_TYPES]
    )
].copy()

sort_cols = [c for c in ["session_type", "subject_id", "session_date"] if c in cohort_df.columns]
cohort_df = cohort_df.sort_values(sort_cols).reset_index(drop=True)

if MAX_SESSIONS is not None:
    cohort_df = cohort_df.iloc[:int(MAX_SESSIONS)].copy()

if cohort_df.empty:
    raise ValueError(
        f"No sessions matched TARGET_SESSION_TYPES={TARGET_SESSION_TYPES}."
    )

show_cols = [c for c in [
    "session_id", "subject_id", "session_date", "session_type",
    "dmd1_depth", "dmd2_depth", "quality",
] if c in cohort_df.columns]

display(cohort_df[show_cols])
print(f"Matched {len(cohort_df)} sessions from {cohort_df['subject_id'].nunique()} mice.")

# A duplicated subject/session-type is not necessarily wrong, but it is worth seeing.
dup = cohort_df.duplicated(["subject_id", "session_type"], keep=False)
if dup.any():
    warnings.warn(
        "At least one mouse has >1 matching session for the same session_type. "
        "All matching sessions will be included; session-level clustering is preserved."
    )
    display(cohort_df.loc[dup, show_cols])

session_assets = []
for _, row in cohort_df.iterrows():
    session_assets.append((row, registry.resolve_assets(row)))


## 2. Trace loading and depth-bin helpers

Depth bins are **half-open**: a cell at exactly 100 µm is assigned to `100–200 µm`, not `0–100 µm`.

Batch processing is intentionally fail-soft: a malformed session is recorded in `failures_df` rather than discarding the rest of the cohort, unless `FAIL_FAST=True`.


In [ ]:
def _as_scalar(x):
    arr = np.asarray(x).squeeze()
    if arr.size == 1:
        value = arr.reshape(-1)[0]
        return value.item() if isinstance(value, np.generic) else value
    return arr


def depth_bin_fields(depth_um, bin_um=100):
    depth_um = float(depth_um)
    start = math.floor(depth_um / bin_um) * bin_um
    stop = start + bin_um
    return {
        "depth_bin_start_um": float(start),
        "depth_bin_stop_um": float(stop),
        "depth_bin": f"{start:g}–{stop:g} µm",
    }


def _orient_trace_dataset(ds, expected_n_rois):
    arr = np.asarray(ds[()], dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D trace dataset, got {arr.shape}")
    if arr.shape[0] == expected_n_rois:
        return arr
    if arr.shape[1] == expected_n_rois:
        return arr.T
    raise ValueError(
        f"Neither axis matches expected_n_rois={expected_n_rois}; shape={arr.shape}"
    )


def _trial_slices_from_lengths(trial_lengths):
    """Build concatenated-trace trial slices from processed H5 bookkeeping."""
    trial_lengths = np.asarray(trial_lengths, dtype=int).reshape(-1)
    trial_slices = []
    cursor = 0
    for i, n in enumerate(trial_lengths, start=1):
        n = int(n)
        if n < 0:
            raise ValueError(f"Negative processed trial length for trial {i}: {n}")
        trial_slices.append({
            "trial": i,
            "start": cursor,
            "stop": cursor + n,
        })
        cursor += n
    return trial_slices


def load_derived_voltage_traces(summary_path, trace_h5_path, fs, signal="dff"):
    """Load processed voltage-session H5 data as DMD -> ROI x time.

    The processed session H5 is authoritative for the processed time axis.
    In the voltage postprocessor, overlong source epochs are clipped at their
    paired behavior endpoints before these datasets are written. Therefore
    ``summary/trialLineRanges/trialGlobalNLines`` describes the source extraction
    and is not required to equal the processed H5 sample count.

    The summary is used here only for source/ROI metadata. Processed trial
    lengths, timebase, and epoch labels are read directly from the H5 that
    contains the analyzed trace.
    """
    with h5py.File(summary_path, "r") as sf:
        n_rois = np.asarray(
            sf["summary/nAnalysisROIs"][()],
            dtype=int,
        ).reshape(-1)

        summary_trial_lengths = None
        if "summary/trialLineRanges/trialGlobalNLines" in sf:
            summary_trial_lengths = np.asarray(
                sf["summary/trialLineRanges/trialGlobalNLines"][()],
                dtype=int,
            ).reshape(-1)

    traces = {}
    processed_meta = {}

    with h5py.File(trace_h5_path, "r") as tf:
        for dmd, expected_n_rois in enumerate(n_rois, start=1):
            dmd_key = f"DMD{dmd}"
            if dmd_key not in tf:
                raise KeyError(
                    f"{dmd_key!r} not found in {trace_h5_path}. "
                    f"Root groups: {list(tf.keys())}"
                )

            grp = tf[dmd_key]
            if signal not in grp:
                raise KeyError(
                    f"{dmd_key}/{signal!s} not found in {trace_h5_path}. "
                    f"Available datasets: {list(grp.keys())}"
                )

            trace = _orient_trace_dataset(grp[signal], int(expected_n_rois))
            traces[dmd] = trace
            n_samples = int(trace.shape[1])

            if "trial_lengths_samples" not in grp:
                raise KeyError(
                    f"{dmd_key}/trial_lengths_samples is required for processed "
                    "session-trace analysis."
                )
            trial_lengths = np.asarray(
                grp["trial_lengths_samples"][:],
                dtype=int,
            ).reshape(-1)

            if int(trial_lengths.sum()) != n_samples:
                raise ValueError(
                    f"{dmd_key}: processed trial lengths sum to "
                    f"{trial_lengths.sum():,}, but {signal} contains "
                    f"{n_samples:,} samples."
                )

            if "timebase_sec" not in grp:
                raise KeyError(
                    f"{dmd_key}/timebase_sec is required for processed "
                    "session-trace analysis."
                )
            timebase_sec = np.asarray(
                grp["timebase_sec"][:],
                dtype=np.float64,
            ).reshape(-1)
            if timebase_sec.size != n_samples:
                raise ValueError(
                    f"{dmd_key}: timebase_sec has {timebase_sec.size:,} samples, "
                    f"but {signal} contains {n_samples:,}."
                )

            sample_epoch = (
                np.asarray(grp["sample_epoch"][:], dtype=int).reshape(-1)
                if "sample_epoch" in grp else None
            )
            if sample_epoch is not None and sample_epoch.size != n_samples:
                raise ValueError(
                    f"{dmd_key}: sample_epoch has {sample_epoch.size:,} samples, "
                    f"but {signal} contains {n_samples:,}."
                )

            trial_starts_sec = (
                np.asarray(grp["trial_starts_sec"][:], dtype=np.float64).reshape(-1)
                if "trial_starts_sec" in grp else None
            )
            trial_valid_mask = (
                np.asarray(grp["trial_valid_mask"][:], dtype=bool).reshape(-1)
                if "trial_valid_mask" in grp else None
            )

            processed_meta[dmd] = {
                "trial_lengths_samples": trial_lengths,
                "timebase_sec": timebase_sec,
                "sample_epoch": sample_epoch,
                "trial_starts_sec": trial_starts_sec,
                "trial_valid_mask": trial_valid_mask,
            }

    sample_counts = {x.shape[1] for x in traces.values()}
    if len(sample_counts) != 1:
        raise ValueError(f"DMD datasets have different sample counts: {sample_counts}")
    n_samples = int(sample_counts.pop())

    # The session-trace H5 is produced from the same reconstructed acquisition
    # for each DMD. Require identical processed temporal bookkeeping before
    # performing cross-DMD synchrony analyses.
    ref_dmd = sorted(processed_meta)[0]
    ref = processed_meta[ref_dmd]

    for dmd in sorted(processed_meta)[1:]:
        meta = processed_meta[dmd]
        if not np.array_equal(
            ref["trial_lengths_samples"],
            meta["trial_lengths_samples"],
        ):
            raise ValueError(
                f"DMD{ref_dmd} and DMD{dmd} disagree on processed "
                "trial_lengths_samples; refusing cross-DMD temporal analysis."
            )
        if not np.allclose(
            ref["timebase_sec"],
            meta["timebase_sec"],
            rtol=0,
            atol=max(1e-9, 0.05 / float(fs)),
            equal_nan=True,
        ):
            raise ValueError(
                f"DMD{ref_dmd} and DMD{dmd} disagree on processed timebase_sec; "
                "refusing cross-DMD temporal analysis."
            )
        if (
            ref["sample_epoch"] is not None
            and meta["sample_epoch"] is not None
            and not np.array_equal(ref["sample_epoch"], meta["sample_epoch"])
        ):
            raise ValueError(
                f"DMD{ref_dmd} and DMD{dmd} disagree on sample_epoch."
            )

    trial_lengths = ref["trial_lengths_samples"]
    trial_slices = _trial_slices_from_lengths(trial_lengths)

    # Keep a compressed sample-rate time vector for within-trace descriptive
    # quantities, while retaining the extractor's behavior/HARP-aligned timebase
    # separately for absolute acquisition timing.
    compressed_time_sec = np.arange(n_samples, dtype=np.float64) / float(fs)
    processed_timebase_sec = np.asarray(ref["timebase_sec"], dtype=np.float64)
    acquisition_time_sec = processed_timebase_sec - processed_timebase_sec[0]

    source_n_samples = (
        int(summary_trial_lengths.sum())
        if summary_trial_lengths is not None else None
    )
    clipped_n_samples = (
        source_n_samples - n_samples
        if source_n_samples is not None else None
    )

    if clipped_n_samples is not None and clipped_n_samples > 0:
        print(
            f"    Processed H5 is {clipped_n_samples:,} samples "
            f"({clipped_n_samples / float(fs):.2f} s) shorter than the source "
            "summary; using processed H5 timing/bookkeeping (expected for "
            "behavior-epoch endpoint clipping)."
        )
    elif clipped_n_samples is not None and clipped_n_samples < 0:
        warnings.warn(
            "Processed H5 contains more samples than the source summary. "
            "This is not explained by normal epoch-end clipping and should be "
            "inspected."
        )

    return traces, {
        "mode": "processed_trial_concatenated",
        "signal": signal,
        "trial_lengths_samples": trial_lengths,
        "summary_trial_lengths_samples": summary_trial_lengths,
        "source_n_samples": source_n_samples,
        "processed_n_samples": n_samples,
        "n_samples_clipped_vs_summary": clipped_n_samples,
        "compressed_time_sec": compressed_time_sec,
        "acquisition_time_sec": acquisition_time_sec,
        "processed_timebase_sec": processed_timebase_sec,
        "sample_epoch": ref["sample_epoch"],
        "trial_starts_sec": ref["trial_starts_sec"],
        "trial_valid_mask": ref["trial_valid_mask"],
        "trial_slices": trial_slices,
    }


def resolve_session_inputs(asset):
    voltage_qc_path = (
        asset.qc_dir / "voltage" / f"voltage_extraction_qc_{TRACE_VARIANT}.json"
    )
    with open(voltage_qc_path, "r") as f:
        voltage_qc = json.load(f)

    fs = float(voltage_qc["sample_rate_hz"])
    summary_path = Path(voltage_qc["summary_mat"])
    trace_h5_path = (
        asset.derived_dir / "voltage" / f"voltage_session_traces_{TRACE_VARIANT}.h5"
    )
    return fs, summary_path, trace_h5_path


## 3. Spike/burst detector

The detector is intentionally kept close to the original notebook so that the cohort rewrite does not change the biological definition while changing the sampling scheme. The main analytical change here is **aggregation**, not spike-call criteria.

For cohort comparisons, consider a separate detector-sensitivity pass later: thresholds based on whole-trace SD can depend modestly on a cell's firing/burst rate. The notebook therefore keeps detection QC outputs available at the per-session level rather than silently changing thresholds.


In [ ]:
DETECTION = {
    "candidate_height_sd": 1.5,
    "waveform_height_sd": 2.0,
    "prominence_dff": 0.10,
    "prominence_window_ms": 50.0,
    "refractory_ms": 1.0,
    "min_width_ms": 0.1,

    "group_link_ms": 20.0,
    "burst_min_spikes": 3,

    "burst_envelope_smooth_ms": 3.0,
    "burst_envelope_height_sd": 1.0,
    "burst_envelope_min_ms": 10.0,
    "burst_envelope_merge_gap_ms": 5.0,
    "burst_expand_pre_max_ms": 10.0,
    "burst_expand_post_max_ms": 50.0,

    "waveform_pre_ms": 10.0,
    "waveform_post_ms": 20.0,
    "waveform_baseline_ms": (-3.5, -1.0),

    "burst_example_pre_ms": 10.0,
    "burst_example_post_ms": 100.0,

    "plateau_window_ms": (8.0, 50.0),
    "baseline_bin_s": 0.25,
    "baseline_percentile": 20,
    "baseline_smooth_s": 2.0,

    # Use > group_link_ms so concatenation boundaries cannot create burst groups or fast synchrony artifacts.
    "exclude_trial_edge_ms": 25.0,
}

DETECTION


In [ ]:
def fill_nonfinite(y):
    y = np.asarray(y, dtype=np.float32)
    good = np.isfinite(y)
    if good.all():
        return y
    if not good.any():
        raise ValueError("Trace contains no finite samples.")
    out = y.copy()
    index = np.arange(len(y))
    out[~good] = np.interp(index[~good], index[good], y[good])
    return out


def robust_mad(x):
    x = np.asarray(x, dtype=float)
    center = np.nanmedian(x)
    return float(1.4826 * np.nanmedian(np.abs(x - center)))


def quantile_baseline(y, fs, bin_s, percentile, smooth_s):
    y = np.asarray(y, dtype=np.float32)
    bin_samples = max(1, int(round(bin_s * fs)))
    n_full = y.size // bin_samples

    values = []
    centers = []
    if n_full:
        blocks = y[:n_full * bin_samples].reshape(n_full, bin_samples)
        values.extend(np.nanpercentile(blocks, percentile, axis=1))
        centers.extend((np.arange(n_full) + 0.5) * bin_samples - 0.5)
    if n_full * bin_samples < y.size:
        values.append(np.nanpercentile(y[n_full * bin_samples:], percentile))
        centers.append(0.5 * (n_full * bin_samples + y.size - 1))

    values = np.asarray(values, dtype=np.float32)
    centers = np.asarray(centers, dtype=float)

    if len(values) > 3:
        values = ndimage.gaussian_filter1d(
            values,
            sigma=max(0.5, smooth_s / bin_s),
            mode="nearest",
            truncate=3,
        )

    return np.interp(np.arange(y.size), centers, values).astype(np.float32)


def split_peak_indices(peaks, max_gap_samples):
    peaks = np.asarray(peaks, dtype=int)
    if peaks.size == 0:
        return []
    cuts = np.flatnonzero(np.diff(peaks) > max_gap_samples) + 1
    return list(np.split(np.arange(len(peaks), dtype=int), cuts))


def mask_to_intervals(mask):
    changes = np.diff(np.asarray(mask, dtype=np.int8), prepend=0, append=0)
    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1)
    return [(int(start), int(stop)) for start, stop in zip(starts, stops)]


def merge_intervals(intervals, max_gap_samples):
    if not intervals:
        return []
    intervals = sorted(intervals)
    merged = [list(intervals[0])]
    for start, stop in intervals[1:]:
        if start - merged[-1][1] <= max_gap_samples:
            merged[-1][1] = max(merged[-1][1], stop)
        else:
            merged.append([start, stop])
    return [tuple(x) for x in merged]


def local_crossing_metrics(waveform, peak_index, fs):
    amplitude = waveform[peak_index]
    if not np.isfinite(amplitude) or amplitude <= 0:
        return np.nan, np.nan, np.nan

    def left_cross(frac):
        target = frac * amplitude
        for i in range(peak_index - 1, -1, -1):
            if waveform[i] <= target < waveform[i + 1]:
                return i + (
                    (target - waveform[i])
                    / (waveform[i + 1] - waveform[i])
                )
        return np.nan

    def right_cross(frac):
        target = frac * amplitude
        for i in range(peak_index, len(waveform) - 1):
            if waveform[i] >= target > waveform[i + 1]:
                return i + (
                    (waveform[i] - target)
                    / (waveform[i] - waveform[i + 1])
                )
        return np.nan

    left10 = left_cross(0.10)
    left50 = left_cross(0.50)
    left90 = left_cross(0.90)
    right50 = right_cross(0.50)
    right10 = right_cross(0.10)

    rise_ms = (
        (left90 - left10) / fs * 1000
        if np.isfinite(left10 + left90)
        else np.nan
    )
    width_ms = (
        (right50 - left50) / fs * 1000
        if np.isfinite(left50 + right50)
        else np.nan
    )
    decay_ms = (
        (right10 - peak_index) / fs * 1000
        if np.isfinite(right10)
        else np.nan
    )
    return rise_ms, width_ms, decay_ms


def find_depolarization_episodes(x, fs, params):
    sigma = max(
        0.1,
        params["burst_envelope_smooth_ms"] / 1000 * fs,
    )
    envelope = ndimage.gaussian_filter1d(
        x,
        sigma=sigma,
        mode="nearest",
        truncate=3,
    )
    threshold = (
        np.nanmean(envelope)
        + params["burst_envelope_height_sd"] * np.nanstd(envelope, ddof=1)
    )

    intervals = mask_to_intervals(envelope > threshold)
    intervals = merge_intervals(
        intervals,
        max_gap_samples=int(round(
            params["burst_envelope_merge_gap_ms"] / 1000 * fs
        )),
    )
    min_samples = int(round(
        params["burst_envelope_min_ms"] / 1000 * fs
    ))
    intervals = [
        (start, stop)
        for start, stop in intervals
        if stop - start >= min_samples
    ]
    return intervals, envelope, float(threshold)


def expand_burst_with_envelope(group, episodes, fs, params, n_samples):
    first = int(group[0])
    last = int(group[-1])

    overlaps = [
        (start, stop)
        for start, stop in episodes
        if stop > first and start <= last
    ]

    start = first
    stop = last + 1
    if overlaps:
        start = min(start, min(x[0] for x in overlaps))
        stop = max(stop, max(x[1] for x in overlaps))

    pre_limit = int(round(
        params["burst_expand_pre_max_ms"] / 1000 * fs
    ))
    post_limit = int(round(
        params["burst_expand_post_max_ms"] / 1000 * fs
    ))

    start = max(first - pre_limit, start, 0)
    stop = min(last + post_limit + 1, stop, n_samples)
    return int(start), int(stop)


In [ ]:
def detect_and_characterize_roi(
    y,
    fs,
    params,
    trial_slices=None,
):
    """Detect candidate spikes and classify singleton, doublet, and burst events."""
    y = fill_nonfinite(y)
    center = float(np.nanmean(y))
    scale = float(np.nanstd(y, ddof=1))

    distance = max(
        1,
        int(round(params["refractory_ms"] / 1000 * fs)),
    )
    min_width = max(
        1.0,
        params["min_width_ms"] / 1000 * fs,
    )
    prominence_wlen = max(
        3,
        int(round(params["prominence_window_ms"] / 1000 * fs)),
    )
    if prominence_wlen % 2 == 0:
        prominence_wlen += 1

    peaks, properties = signal.find_peaks(
        y,
        height=center + params["candidate_height_sd"] * scale,
        prominence=params["prominence_dff"],
        distance=distance,
        width=min_width,
        wlen=prominence_wlen,
    )

    edge_ms = params["exclude_trial_edge_ms"]
    if edge_ms > 0 and trial_slices:
        edge = int(round(edge_ms / 1000 * fs))
        keep = np.ones(len(peaks), dtype=bool)
        boundaries = np.asarray(
            [sl.stop for sl in trial_slices[:-1]],
            dtype=int,
        )
        for boundary in boundaries:
            keep &= np.abs(peaks - boundary) > edge
        peaks = peaks[keep]
        properties = {
            key: np.asarray(value)[keep]
            for key, value in properties.items()
        }

    high_confidence = (
        properties.get("peak_heights", np.array([]))
        >= center + params["waveform_height_sd"] * scale
    )

    group_gap = int(round(params["group_link_ms"] / 1000 * fs))
    group_indices = split_peak_indices(peaks, group_gap)

    baseline = quantile_baseline(
        y,
        fs,
        params["baseline_bin_s"],
        params["baseline_percentile"],
        params["baseline_smooth_s"],
    )
    x = y - baseline

    episodes, burst_envelope, burst_envelope_threshold = (
        find_depolarization_episodes(x, fs, params)
    )

    noise_residual = y - ndimage.gaussian_filter1d(
        y,
        sigma=max(0.1, 2.0 / 1000 * fs),
        mode="nearest",
        truncate=3,
    )
    noise = robust_mad(noise_residual)
    if not np.isfinite(noise) or noise <= 0:
        noise = float(np.nanstd(noise_residual, ddof=1))

    event_id = np.full(len(peaks), -1, dtype=int)
    spike_order = np.zeros(len(peaks), dtype=int)
    spike_class = np.full(len(peaks), "", dtype=object)

    event_rows = []
    burst_waveforms = []
    burst_relative_spikes_ms = []
    burst_event_ids = []

    burst_pre = int(round(
        params["burst_example_pre_ms"] / 1000 * fs
    ))
    burst_post = int(round(
        params["burst_example_post_ms"] / 1000 * fs
    ))

    for current_event_id, indices in enumerate(group_indices):
        group = peaks[indices]
        n_group = len(group)

        if n_group == 1:
            event_class = "singleton"
        elif n_group == 2:
            event_class = "doublet"
        else:
            event_class = "burst"

        event_id[indices] = current_event_id
        spike_order[indices] = np.arange(1, n_group + 1)
        spike_class[indices] = event_class

        first = int(group[0])
        last = int(group[-1])

        if event_class == "burst":
            onset, offset = expand_burst_with_envelope(
                group,
                episodes,
                fs,
                params,
                len(y),
            )
        else:
            onset, offset = first, last + 1

        event_rows.append({
            "event_id": current_event_id,
            "event_class": event_class,
            "event_onset_sample": onset,
            "first_spike_sample": first,
            "event_peak_sample": int(group[np.argmax(x[group])]),
            "last_spike_sample": last,
            "event_offset_sample": offset,
            "n_inferred_spikes": n_group,
            "spike_train_duration_ms": (last - first) / fs * 1000,
            "event_window_duration_ms": (offset - onset) / fs * 1000,
        })

        if (
            event_class == "burst"
            and first - burst_pre >= 0
            and first + burst_post < len(x)
        ):
            waveform = x[
                first - burst_pre:first + burst_post + 1
            ].copy()
            pre_segment = waveform[
                :max(3, int(round(0.6 * burst_pre)))
            ]
            waveform -= np.nanmedian(pre_segment)
            burst_waveforms.append(waveform)
            burst_relative_spikes_ms.append(
                (group - first) / fs * 1000
            )
            burst_event_ids.append(current_event_id)

    event_columns = [
        "event_id",
        "event_class",
        "event_onset_sample",
        "first_spike_sample",
        "event_peak_sample",
        "last_spike_sample",
        "event_offset_sample",
        "n_inferred_spikes",
        "spike_train_duration_ms",
        "event_window_duration_ms",
    ]
    events_df = pd.DataFrame(event_rows, columns=event_columns)

    plateau0 = int(round(
        params["plateau_window_ms"][0] / 1000 * fs
    ))
    plateau1 = int(round(
        params["plateau_window_ms"][1] / 1000 * fs
    ))
    plateau_index = np.full(len(peaks), np.nan)

    for i, peak in enumerate(peaks):
        amplitude = x[peak]
        if amplitude > 0 and peak + plateau1 <= len(x):
            plateau_index[i] = (
                np.nanmean(x[peak + plateau0:peak + plateau1])
                / amplitude
            )

    waveform_pre = int(round(
        params["waveform_pre_ms"] / 1000 * fs
    ))
    waveform_post = int(round(
        params["waveform_post_ms"] / 1000 * fs
    ))
    baseline_start = int(round(
        (params["waveform_baseline_ms"][0]
         + params["waveform_pre_ms"])
        / 1000 * fs
    ))
    baseline_stop = int(round(
        (params["waveform_baseline_ms"][1]
         + params["waveform_pre_ms"])
        / 1000 * fs
    ))

    amplitude_dff = np.full(len(peaks), np.nan)
    snr = np.full(len(peaks), np.nan)
    rise_ms = np.full(len(peaks), np.nan)
    width_ms = np.full(len(peaks), np.nan)
    decay_ms = np.full(len(peaks), np.nan)
    is_waveform_spike = np.zeros(len(peaks), dtype=bool)

    isolated_waveforms = []
    isolated_spike_indices = []

    for i, peak in enumerate(peaks):
        if spike_class[i] != "singleton" or not high_confidence[i]:
            continue
        if peak - waveform_pre < 0 or peak + waveform_post >= len(y):
            continue

        waveform = y[
            peak - waveform_pre:peak + waveform_post + 1
        ].copy()
        local_baseline = np.nanmedian(
            waveform[baseline_start:baseline_stop]
        )
        waveform -= local_baseline

        amplitude = waveform[waveform_pre]
        if not np.isfinite(amplitude) or amplitude <= 0:
            continue

        rise, width, decay = local_crossing_metrics(
            waveform,
            waveform_pre,
            fs,
        )
        amplitude_dff[i] = amplitude
        snr[i] = amplitude / noise if noise > 0 else np.nan
        rise_ms[i] = rise
        width_ms[i] = width
        decay_ms[i] = decay
        is_waveform_spike[i] = True
        isolated_waveforms.append(waveform)
        isolated_spike_indices.append(i)

    spike_rows = []
    for i, peak in enumerate(peaks):
        spike_rows.append({
            "spike_sample": int(peak),
            "event_id": int(event_id[i]),
            "spike_order": int(spike_order[i]),
            "spike_class": str(spike_class[i]),
            "is_high_confidence": bool(high_confidence[i]),
            "is_waveform_spike": bool(is_waveform_spike[i]),
            "is_burst_spike": bool(spike_class[i] == "burst"),
            "peak_height_dff": float(
                properties["peak_heights"][i]
            ),
            "prominence_dff": float(
                properties["prominences"][i]
            ),
            "find_peaks_width_ms": float(
                properties["widths"][i] / fs * 1000
            ),
            "amplitude_dff": float(amplitude_dff[i]),
            "snr": float(snr[i]),
            "plateau_index": float(plateau_index[i]),
            "rise10_90_ms": float(rise_ms[i]),
            "width50_ms": float(width_ms[i]),
            "decay90_10_ms": float(decay_ms[i]),
        })

    spike_columns = [
        "spike_sample",
        "event_id",
        "spike_order",
        "spike_class",
        "is_high_confidence",
        "is_waveform_spike",
        "is_burst_spike",
        "peak_height_dff",
        "prominence_dff",
        "find_peaks_width_ms",
        "amplitude_dff",
        "snr",
        "plateau_index",
        "rise10_90_ms",
        "width50_ms",
        "decay90_10_ms",
    ]
    spikes_df = pd.DataFrame(spike_rows, columns=spike_columns)

    waveform_length = waveform_pre + waveform_post + 1
    if isolated_waveforms:
        isolated_waveforms = np.asarray(
            isolated_waveforms,
            dtype=np.float32,
        )
    else:
        isolated_waveforms = np.empty(
            (0, waveform_length),
            dtype=np.float32,
        )

    burst_length = burst_pre + burst_post + 1
    if burst_waveforms:
        burst_waveforms = np.asarray(
            burst_waveforms,
            dtype=np.float32,
        )
    else:
        burst_waveforms = np.empty(
            (0, burst_length),
            dtype=np.float32,
        )

    return {
        "spikes": spikes_df,
        "events": events_df,
        "spike_samples": peaks,
        "isolated_waveforms": isolated_waveforms,
        "isolated_spike_indices": np.asarray(
            isolated_spike_indices,
            dtype=int,
        ),
        "burst_waveforms": burst_waveforms,
        "burst_relative_spikes_ms": burst_relative_spikes_ms,
        "burst_event_ids": np.asarray(burst_event_ids, dtype=int),
        # Full-session baseline/envelope arrays are intentionally not returned in
        # cohort mode; keeping them for every cell would dominate memory.
        "burst_envelope_threshold": burst_envelope_threshold,
        "candidate_height_threshold": (
            center + params["candidate_height_sd"] * scale
        ),
        "waveform_height_threshold": (
            center + params["waveform_height_sd"] * scale
        ),
        "trace_mean": center,
        "trace_sd": scale,
        "noise": noise,
    }


## 4. Waveform, compact-example, and synchrony helpers

Isolated optical spike waveforms are z-scored using the **pre-spike baseline** of each event. This makes the waveform panel interpretable in local-noise units while preserving cell-to-cell differences in spike shape.


In [ ]:
def baseline_zscore_waveforms(waveforms, waveform_time_ms, baseline_ms=(-9.0, -3.0)):
    waveforms = np.asarray(waveforms, dtype=np.float32)
    if waveforms.ndim != 2 or len(waveforms) == 0:
        return np.empty_like(waveforms)

    mask = (
        (waveform_time_ms >= baseline_ms[0])
        & (waveform_time_ms <= baseline_ms[1])
    )
    if mask.sum() < 3:
        raise ValueError("Waveform baseline window contains fewer than 3 samples.")

    center = np.nanmean(waveforms[:, mask], axis=1)
    scale = np.nanstd(waveforms[:, mask], axis=1, ddof=1)
    keep = np.isfinite(scale) & (scale > 0)

    z = np.full_like(waveforms, np.nan, dtype=np.float32)
    z[keep] = (
        waveforms[keep] - center[keep, None]
    ) / scale[keep, None]
    return z


def select_representative_bursts(result, n_show=5):
    waveforms = result["burst_waveforms"]
    relative_spikes = result["burst_relative_spikes_ms"]
    event_ids = result["burst_event_ids"]
    events = result["events"].set_index("event_id")

    if len(waveforms) == 0:
        return []

    ordering_values = np.asarray([
        events.loc[event_id, "n_inferred_spikes"]
        + events.loc[event_id, "event_window_duration_ms"] / 100
        for event_id in event_ids
    ])
    order = np.argsort(ordering_values)
    if len(order) <= n_show:
        selected = order
    else:
        selected = order[np.linspace(0, len(order) - 1, n_show, dtype=int)]

    examples = []
    for index in selected:
        event_id = int(event_ids[index])
        event = events.loc[event_id]
        examples.append({
            "waveform": np.asarray(waveforms[index], dtype=np.float32),
            "relative_spikes_ms": np.asarray(relative_spikes[index], dtype=float),
            "n_inferred_spikes": int(event["n_inferred_spikes"]),
            "event_window_duration_ms": float(event["event_window_duration_ms"]),
        })
    return examples


def fraction_spikes_near(a, b, dt):
    if len(a) == 0 or len(b) == 0:
        return np.nan
    b = np.sort(np.asarray(b))
    hits = 0
    for time in np.asarray(a):
        index = np.searchsorted(b, time)
        near = (
            (index < len(b) and abs(b[index] - time) <= dt)
            or (index > 0 and abs(b[index - 1] - time) <= dt)
        )
        hits += near
    return hits / len(a)


def fraction_time_tiled(times, dt, t_start, t_stop):
    if len(times) == 0:
        return 0.0
    intervals = np.c_[
        np.maximum(np.asarray(times) - dt, t_start),
        np.minimum(np.asarray(times) + dt, t_stop),
    ]
    intervals = intervals[np.argsort(intervals[:, 0])]
    covered = 0.0
    start, stop = intervals[0]
    for new_start, new_stop in intervals[1:]:
        if new_start <= stop:
            stop = max(stop, new_stop)
        else:
            covered += stop - start
            start, stop = new_start, new_stop
    covered += stop - start
    return covered / (t_stop - t_start)


def sttc(a, b, dt, t_start, t_stop):
    if len(a) == 0 or len(b) == 0:
        return np.nan
    pa = fraction_spikes_near(a, b, dt)
    pb = fraction_spikes_near(b, a, dt)
    ta = fraction_time_tiled(a, dt, t_start, t_stop)
    tb = fraction_time_tiled(b, dt, t_start, t_stop)
    term_a = (pa - tb) / (1 - pa * tb + 1e-12)
    term_b = (pb - ta) / (1 - pb * ta + 1e-12)
    return 0.5 * (term_a + term_b)


def binned_rate(times, duration, bin_s):
    edges = np.arange(0, duration + bin_s, bin_s)
    counts, _ = np.histogram(times, bins=edges)
    return counts / bin_s


def safe_correlation(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    if len(a) < 2 or len(b) < 2 or np.nanstd(a) == 0 or np.nanstd(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def interval_jaccard(intervals_a, intervals_b):
    """Exact temporal Jaccard overlap for two sets of non-overlapping intervals."""
    a = sorted((float(x), float(y)) for x, y in intervals_a if y > x)
    b = sorted((float(x), float(y)) for x, y in intervals_b if y > x)
    if not a and not b:
        return np.nan

    len_a = sum(y - x for x, y in a)
    len_b = sum(y - x for x, y in b)
    i = j = 0
    intersection = 0.0
    while i < len(a) and j < len(b):
        left = max(a[i][0], b[j][0])
        right = min(a[i][1], b[j][1])
        if right > left:
            intersection += right - left
        if a[i][1] <= b[j][1]:
            i += 1
        else:
            j += 1
    union = len_a + len_b - intersection
    return intersection / union if union > 0 else np.nan


def cross_correlogram(a, b, window_s=0.050, bin_s=0.001):
    differences = []
    b = np.sort(np.asarray(b))
    for time in np.asarray(a):
        low = np.searchsorted(b, time - window_s)
        high = np.searchsorted(b, time + window_s, side="right")
        differences.extend(b[low:high] - time)
    edges = np.arange(-window_s, window_s + bin_s, bin_s)
    counts, _ = np.histogram(differences, bins=edges)
    centers = edges[:-1] + bin_s / 2
    conditional_rate = counts / (max(len(a), 1) * bin_s)
    return centers, conditional_rate


def unit_key(subject_id, session_id, dmd, roi):
    return (str(subject_id), str(session_id), int(dmd), int(roi))


def pair_bin_label(a_label, a_start, b_label, b_start):
    if a_start <= b_start:
        return f"{a_label} ↔ {b_label}", a_start, b_start
    return f"{b_label} ↔ {a_label}", b_start, a_start


## 5. Session analyzer

This function loads one asset, detects spikes ROI-by-ROI, stores compact waveform/trace examples, computes ROI metrics, and then constructs **all pairwise synchrony rows within that session only**.

Event-level spike and burst tables are written per session and discarded from memory by default, which keeps the cohort pass tractable for many mice.


In [ ]:
def analyze_session(session_row, asset):
    fs, summary_path, trace_h5_path = resolve_session_inputs(asset)
    traces, trace_info = load_derived_voltage_traces(
        summary_path,
        trace_h5_path,
        fs,
        signal="dff",
    )

    segment_slices = [
        slice(x["start"], x["stop"])
        for x in trace_info["trial_slices"]
    ]
    duration_s = next(iter(traces.values())).shape[1] / fs
    compressed_time = trace_info["compressed_time_sec"]
    acquisition_time = trace_info["acquisition_time_sec"]

    subject_id = asset.subject_id
    session_id = asset.session_id
    session_type = str(session_row["session_type"])

    depth_by_dmd = {}
    for dmd in traces:
        key = f"dmd{dmd}_depth"
        if key not in asset.metadata:
            raise KeyError(f"{key!r} missing from asset.metadata for {session_id}")
        depth_by_dmd[dmd] = float(asset.metadata[key])

    waveform_pre = int(round(DETECTION["waveform_pre_ms"] / 1000 * fs))
    waveform_post = int(round(DETECTION["waveform_post_ms"] / 1000 * fs))
    waveform_time_ms = np.arange(-waveform_pre, waveform_post + 1) / fs * 1000

    burst_pre = int(round(DETECTION["burst_example_pre_ms"] / 1000 * fs))
    burst_post = int(round(DETECTION["burst_example_post_ms"] / 1000 * fs))
    burst_time_ms = np.arange(-burst_pre, burst_post + 1) / fs * 1000

    metric_rows = []
    session_units = []
    unit_compact = {}
    session_event_tables = []
    session_spike_tables = []

    trace0, trace1 = TRACE_EXAMPLE_WINDOW_SEC
    trace_i0 = max(0, int(round(trace0 * fs)))
    trace_i1 = min(next(iter(traces.values())).shape[1], int(round(trace1 * fs)))

    for dmd, x in traces.items():
        depth_um = depth_by_dmd[dmd]
        depth_fields = depth_bin_fields(depth_um, DEPTH_BIN_UM)

        for roi, y in enumerate(x):
            result = detect_and_characterize_roi(
                y,
                fs,
                DETECTION,
                trial_slices=segment_slices,
            )

            events = result["events"].assign(
                subject_id=subject_id,
                session_id=session_id,
                session_type=session_type,
                dmd=dmd,
                roi=roi,
                depth_um=depth_um,
                **depth_fields,
            )
            spikes = result["spikes"].assign(
                subject_id=subject_id,
                session_id=session_id,
                session_type=session_type,
                dmd=dmd,
                roi=roi,
                depth_um=depth_um,
                **depth_fields,
            )

            if len(events):
                event_samples = events["event_onset_sample"].to_numpy(dtype=int)
                events["event_time_sec"] = compressed_time[event_samples]
                events["event_acquisition_time_sec"] = acquisition_time[event_samples]
            if len(spikes):
                spike_samples = spikes["spike_sample"].to_numpy(dtype=int)
                spikes["spike_time_sec"] = compressed_time[spike_samples]
                spikes["spike_acquisition_time_sec"] = acquisition_time[spike_samples]

            session_event_tables.append(events)
            session_spike_tables.append(spikes)

            isi = np.diff(result["spike_samples"]) / fs
            cv2 = (
                np.nanmean(2 * np.abs(np.diff(isi)) / (isi[:-1] + isi[1:]))
                if len(isi) > 1 else np.nan
            )

            bursts = events.loc[events["event_class"] == "burst"] if len(events) else events
            singletons = events.loc[events["event_class"] == "singleton"] if len(events) else events
            doublets = events.loc[events["event_class"] == "doublet"] if len(events) else events
            waveform_spikes = spikes.loc[spikes["is_waveform_spike"]] if len(spikes) else spikes

            burst_spike_fraction = (
                np.mean(spikes["spike_class"] == "burst") if len(spikes) else np.nan
            )
            doublet_spike_fraction = (
                np.mean(spikes["spike_class"] == "doublet") if len(spikes) else np.nan
            )
            singleton_spike_fraction = (
                np.mean(spikes["spike_class"] == "singleton") if len(spikes) else np.nan
            )

            burst_intervals_samples = merge_intervals([
                (int(row.event_onset_sample), int(row.event_offset_sample))
                for row in bursts.itertuples()
            ], max_gap_samples=0)
            burst_occupancy = (
                sum(stop - start for start, stop in burst_intervals_samples) / x.shape[1]
                if burst_intervals_samples else 0.0
            )

            metric_rows.append({
                "subject_id": subject_id,
                "session_id": session_id,
                "session_type": session_type,
                "dmd": dmd,
                "roi": roi,
                "label": f"{subject_id} · {session_id} · DMD{dmd} ROI{roi}",
                "depth_um": depth_um,
                **depth_fields,
                "duration_s": duration_s,

                "approx_spike_rate_hz": len(spikes) / duration_s,
                "event_rate_hz": len(events) / duration_s,
                "singleton_event_rate_hz": len(singletons) / duration_s,
                "doublet_event_rate_hz": len(doublets) / duration_s,
                "burst_event_rate_hz": len(bursts) / duration_s,

                "singleton_spike_fraction": singleton_spike_fraction,
                "doublet_spike_fraction": doublet_spike_fraction,
                "spike_burst_fraction": burst_spike_fraction,
                "burst_occupancy": burst_occupancy,

                "median_spikes_per_burst": (
                    bursts["n_inferred_spikes"].median() if len(bursts) else np.nan
                ),
                "median_burst_duration_ms": (
                    bursts["event_window_duration_ms"].median() if len(bursts) else np.nan
                ),
                "median_spike_train_duration_ms": (
                    bursts["spike_train_duration_ms"].median() if len(bursts) else np.nan
                ),
                "isi_cv2": cv2,
                "short_isi_fraction_20ms": (
                    np.mean(isi <= DETECTION["group_link_ms"] / 1000)
                    if len(isi) else np.nan
                ),

                "median_isolated_amplitude_dff": (
                    waveform_spikes["amplitude_dff"].median() if len(waveform_spikes) else np.nan
                ),
                "median_isolated_snr": (
                    waveform_spikes["snr"].median() if len(waveform_spikes) else np.nan
                ),
                "median_isolated_rise10_90_ms": (
                    waveform_spikes["rise10_90_ms"].median() if len(waveform_spikes) else np.nan
                ),
                "median_isolated_width50_ms": (
                    waveform_spikes["width50_ms"].median() if len(waveform_spikes) else np.nan
                ),
                "median_isolated_decay90_10_ms": (
                    waveform_spikes["decay90_10_ms"].median() if len(waveform_spikes) else np.nan
                ),
                "median_plateau_index": (
                    spikes["plateau_index"].median() if len(spikes) else np.nan
                ),

                "n_spikes": len(spikes),
                "n_events": len(events),
                "n_singletons": len(singletons),
                "n_doublets": len(doublets),
                "n_bursts": len(bursts),
                "n_waveform_spikes": len(waveform_spikes),
            })

            spike_times = spikes["spike_time_sec"].to_numpy() if len(spikes) else np.array([])
            isolated_times = (
                waveform_spikes["spike_time_sec"].to_numpy()
                if len(waveform_spikes) else np.array([])
            )
            burst_onset_times = (
                bursts["event_time_sec"].to_numpy() if len(bursts) else np.array([])
            )
            burst_intervals_sec = [
                (start / fs, stop / fs) for start, stop in burst_intervals_samples
            ]

            z_waveforms = baseline_zscore_waveforms(
                result["isolated_waveforms"], waveform_time_ms
            )
            if len(z_waveforms):
                wf_p10, wf_median, wf_p90 = np.nanpercentile(
                    z_waveforms, [10, 50, 90], axis=0
                )
            else:
                wf_p10 = wf_median = wf_p90 = np.full(len(waveform_time_ms), np.nan)

            key = unit_key(subject_id, session_id, dmd, roi)
            compact = {
                "subject_id": str(subject_id),
                "session_id": str(session_id),
                "session_type": session_type,
                "dmd": int(dmd),
                "roi": int(roi),
                "depth_um": float(depth_um),
                **depth_fields,
                "fs": float(fs),
                "duration_s": float(duration_s),
                "spike_times_sec": np.asarray(spike_times, dtype=float),
                "isolated_spike_times_sec": np.asarray(isolated_times, dtype=float),
                "burst_onset_times_sec": np.asarray(burst_onset_times, dtype=float),
                "burst_intervals_sec": burst_intervals_sec,
                "waveform_time_ms": np.asarray(waveform_time_ms, dtype=float),
                "waveform_p10_z": np.asarray(wf_p10, dtype=np.float32),
                "waveform_median_z": np.asarray(wf_median, dtype=np.float32),
                "waveform_p90_z": np.asarray(wf_p90, dtype=np.float32),
                "n_waveform_spikes": int(len(z_waveforms)),
                "burst_time_ms": np.asarray(burst_time_ms, dtype=float),
                "burst_examples": select_representative_bursts(
                    result, N_BURST_EXAMPLES_PER_ROI
                ),
                "trace_window_sec": (
                    np.arange(trace_i0, trace_i1, dtype=float) / fs
                ),
                "trace_window_dff": np.asarray(y[trace_i0:trace_i1], dtype=np.float32).copy(),
            }
            unit_compact[key] = compact
            session_units.append(compact)

    session_roi_metrics = pd.DataFrame(metric_rows)

    # Save heavy event-level data per session, then let it fall out of memory.
    if SAVE_SESSION_EVENT_TABLES:
        session_dir = SESSION_TABLE_DIR / str(session_id)
        session_dir.mkdir(parents=True, exist_ok=True)
        pd.concat(session_spike_tables, ignore_index=True).to_csv(
            session_dir / "detected_spikes.csv", index=False
        )
        pd.concat(session_event_tables, ignore_index=True).to_csv(
            session_dir / "detected_events.csv", index=False
        )
        session_roi_metrics.to_csv(session_dir / "roi_ephys_metrics.csv", index=False)

    pair_rows = []
    for index_a, unit_a in enumerate(session_units):
        for index_b in range(index_a + 1, len(session_units)):
            unit_b = session_units[index_b]

            # The pair is created inside this function, so subject/session identity is guaranteed.
            assert unit_a["subject_id"] == unit_b["subject_id"]
            assert unit_a["session_id"] == unit_b["session_id"]

            pair_label, pair_start_a, pair_start_b = pair_bin_label(
                unit_a["depth_bin"], unit_a["depth_bin_start_um"],
                unit_b["depth_bin"], unit_b["depth_bin_start_um"],
            )

            rate20_a = binned_rate(unit_a["spike_times_sec"], duration_s, 0.020)
            rate20_b = binned_rate(unit_b["spike_times_sec"], duration_s, 0.020)
            rate250_a = binned_rate(unit_a["spike_times_sec"], duration_s, 0.250)
            rate250_b = binned_rate(unit_b["spike_times_sec"], duration_s, 0.250)

            pair_rows.append({
                "subject_id": str(subject_id),
                "session_id": str(session_id),
                "session_type": session_type,
                "unit_a_dmd": unit_a["dmd"],
                "unit_a_roi": unit_a["roi"],
                "unit_b_dmd": unit_b["dmd"],
                "unit_b_roi": unit_b["roi"],
                "unit_a_depth_um": unit_a["depth_um"],
                "unit_b_depth_um": unit_b["depth_um"],
                "unit_a_depth_bin": unit_a["depth_bin"],
                "unit_b_depth_bin": unit_b["depth_bin"],
                "same_dmd": unit_a["dmd"] == unit_b["dmd"],
                "same_depth_bin": unit_a["depth_bin"] == unit_b["depth_bin"],
                "depth_bin_pair": pair_label,
                "depth_bin_pair_start_a": pair_start_a,
                "depth_bin_pair_start_b": pair_start_b,
                "depth_separation_um": abs(unit_a["depth_um"] - unit_b["depth_um"]),
                "mean_pair_depth_um": 0.5 * (unit_a["depth_um"] + unit_b["depth_um"]),

                "spike_sttc_5ms": sttc(
                    unit_a["spike_times_sec"], unit_b["spike_times_sec"],
                    0.005, 0, duration_s,
                ),
                "isolated_spike_sttc_5ms": sttc(
                    unit_a["isolated_spike_times_sec"], unit_b["isolated_spike_times_sec"],
                    0.005, 0, duration_s,
                ),
                "burst_onset_sttc_20ms": sttc(
                    unit_a["burst_onset_times_sec"], unit_b["burst_onset_times_sec"],
                    0.020, 0, duration_s,
                ),
                "spike_count_corr_20ms": safe_correlation(rate20_a, rate20_b),
                "spike_count_corr_250ms": safe_correlation(rate250_a, rate250_b),
                "burst_state_jaccard": interval_jaccard(
                    unit_a["burst_intervals_sec"], unit_b["burst_intervals_sec"]
                ),
            })

    session_pairs = pd.DataFrame(pair_rows)
    session_summary = {
        "subject_id": str(subject_id),
        "session_id": str(session_id),
        "session_type": session_type,
        "sample_rate_hz": fs,
        "duration_s": duration_s,
        "n_cells": len(session_roi_metrics),
        "n_pairs": len(session_pairs),
        "dmd1_depth": depth_by_dmd.get(1, np.nan),
        "dmd2_depth": depth_by_dmd.get(2, np.nan),
    }
    return session_roi_metrics, session_pairs, unit_compact, session_summary


## 6. Run the cohort

This is the expensive cell. Sessions are processed sequentially to limit memory use. A compact unit store retains only spike times, waveform summaries, a short dF/F snippet, and a few representative bursts.


In [ ]:
roi_metric_tables = []
pair_tables = []
unit_store = {}
session_summary_rows = []
failure_rows = []

for idx, (session_row, asset) in enumerate(session_assets, start=1):
    print(
        f"[{idx}/{len(session_assets)}] subject={asset.subject_id} "
        f"session={asset.session_id} type={session_row['session_type']}"
    )
    try:
        metrics_i, pairs_i, compact_i, summary_i = analyze_session(session_row, asset)
        roi_metric_tables.append(metrics_i)
        if len(pairs_i):
            pair_tables.append(pairs_i)
        unit_store.update(compact_i)
        session_summary_rows.append(summary_i)
        print(
            f"    {len(metrics_i)} cells, {len(pairs_i)} within-session pairs"
        )
    except Exception as exc:
        failure_rows.append({
            "subject_id": str(getattr(asset, "subject_id", session_row.get("subject_id", ""))),
            "session_id": str(getattr(asset, "session_id", session_row.get("session_id", ""))),
            "session_type": str(session_row.get("session_type", "")),
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        print(f"    FAILED: {type(exc).__name__}: {exc}")
        if FAIL_FAST:
            raise

if not roi_metric_tables:
    raise RuntimeError("No sessions completed successfully.")

roi_metrics = pd.concat(roi_metric_tables, ignore_index=True)
pair_df = (
    pd.concat(pair_tables, ignore_index=True)
    if pair_tables else pd.DataFrame()
)
session_summary_df = pd.DataFrame(session_summary_rows)
failures_df = pd.DataFrame(failure_rows)

depth_order_df = (
    roi_metrics[["depth_bin", "depth_bin_start_um"]]
    .drop_duplicates()
    .sort_values("depth_bin_start_um")
)
depth_order = depth_order_df["depth_bin"].tolist()
roi_metrics["depth_bin"] = pd.Categorical(
    roi_metrics["depth_bin"], categories=depth_order, ordered=True
)

depth_norm = Normalize(
    vmin=float(depth_order_df["depth_bin_start_um"].min()),
    vmax=float(depth_order_df["depth_bin_start_um"].max() + DEPTH_BIN_UM),
)
depth_colors = {
    row.depth_bin: DEPTH_CMAP(depth_norm(row.depth_bin_start_um + DEPTH_BIN_UM / 2))
    for row in depth_order_df.itertuples()
}

print(
    f"\nCompleted {len(session_summary_df)} sessions / "
    f"{session_summary_df['subject_id'].nunique()} mice / "
    f"{len(roi_metrics)} cells."
)
print(f"Constructed {len(pair_df):,} within-session pairs.")
if len(failures_df):
    display(failures_df)


## 7. Sampling QC: how many cells contribute to each depth bin?

This is not a presentation figure; it is the cohort sanity check to look at before interpreting depth effects.


In [ ]:
cell_counts = (
    roi_metrics.groupby(["depth_bin", "subject_id"], observed=True)
    .size()
    .rename("n_cells")
    .reset_index()
)
summary_counts = (
    roi_metrics.groupby("depth_bin", observed=True)
    .agg(
        n_cells=("roi", "size"),
        n_mice=("subject_id", "nunique"),
        n_sessions=("session_id", "nunique"),
        exact_depths_um=("depth_um", lambda x: sorted(set(map(float, x)))),
    )
    .reindex(depth_order)
)
display(summary_counts)

fig, ax = plt.subplots(figsize=(7.8, 4.6))
rng = np.random.default_rng(1)
for pos, depth_bin in enumerate(depth_order, start=1):
    subset = cell_counts.loc[cell_counts["depth_bin"] == depth_bin]
    if len(subset):
        jitter = rng.normal(0, 0.045, len(subset))
        ax.scatter(
            pos + jitter,
            subset["n_cells"],
            s=55,
            color=depth_colors[depth_bin],
            edgecolor=CHARCOAL,
            linewidth=0.8,
        )
        ax.plot(
            [pos - 0.20, pos + 0.20],
            [subset["n_cells"].median()] * 2,
            color=CHARCOAL,
            lw=2.5,
        )
ax.set_xticks(range(1, len(depth_order) + 1))
ax.set_xticklabels(depth_order)
ax.set_ylabel("Cells per mouse/session")
ax.set_title("Sampling by cortical depth bin")
ax.grid(axis="y", color=LIGHT_GRAY, alpha=0.55)
fig.tight_layout()
save_panel(fig, "00_sampling_by_depth_bin")
plt.show()


# Part I — Spike detection, firing rate, and bursting

The next figure is designed around the slide-10 placeholder: short spike-labelled dF/F examples, spike-rate distributions by depth, and burst fraction versus burst-event frequency.


In [ ]:
def _row_key(row):
    return unit_key(row.subject_id, row.session_id, row.dmd, row.roi)


def representative_row(df, depth_bin, metric="approx_spike_rate_hz", require_bursts=False):
    subset = df.loc[df["depth_bin"] == depth_bin].copy()
    if require_bursts:
        subset = subset.loc[subset["n_bursts"] >= 1]
    subset = subset.loc[np.isfinite(subset[metric])]
    if subset.empty:
        return None
    target = subset[metric].median()
    return subset.iloc[np.argmin(np.abs(subset[metric].to_numpy() - target))]


shallow_bin = depth_order[0]
deep_bin = depth_order[-1]
example_bins = [shallow_bin] if shallow_bin == deep_bin else [shallow_bin, deep_bin]

fig = plt.figure(figsize=(13.8, 7.8))
gs = fig.add_gridspec(2, 2, height_ratios=[1.05, 1.0], hspace=0.36, wspace=0.30)
trace_ax = fig.add_subplot(gs[0, :])
rate_ax = fig.add_subplot(gs[1, 0])
burst_ax = fig.add_subplot(gs[1, 1])

# Short raw dF/F examples from cells near the median firing rate in the shallowest/deepest bins.
trace_offsets = []
current_offset = 0.0
for depth_bin in example_bins:
    row = representative_row(roi_metrics, depth_bin)
    if row is None:
        continue
    unit = unit_store[_row_key(row)]
    t = unit["trace_window_sec"]
    y = unit["trace_window_dff"].astype(float)
    y_centered = y - np.nanmedian(y)
    robust_range = np.nanpercentile(y_centered, 99) - np.nanpercentile(y_centered, 1)
    robust_range = max(float(robust_range), 1e-6)
    offset = current_offset
    current_offset += robust_range * 1.8

    trace_ax.plot(t, y_centered + offset, color=depth_colors[depth_bin], lw=0.85)
    spikes = unit["spike_times_sec"]
    keep = (spikes >= t[0]) & (spikes <= t[-1])
    spikes = spikes[keep]
    if len(spikes):
        values = np.interp(spikes, t, y_centered)
        trace_ax.scatter(
            spikes, values + offset, marker="v", s=22,
            facecolor=CHARCOAL, edgecolor="white", linewidth=0.35,
            zorder=ZORDER["marker"],
        )
    trace_ax.text(
        t[0] - 0.01 * (t[-1] - t[0]), offset,
        f"{depth_bin} · {row.subject_id} · DMD{int(row.dmd)} R{int(row.roi)}\n"
        f"exact depth {row.depth_um:.0f} µm",
        ha="right", va="center", fontsize=9, color=CHARCOAL,
    )

trace_ax.set_yticks([])
trace_ax.set_xlabel("Compressed session time (s)")
trace_ax.set_title("Representative ASAP8 somatic dF/F with detected spikes")

# Cell-level spike-rate distribution with mouse-level medians.
rng = np.random.default_rng(8)
for pos, depth_bin in enumerate(depth_order, start=1):
    subset = roi_metrics.loc[roi_metrics["depth_bin"] == depth_bin]
    values = subset["approx_spike_rate_hz"].dropna().to_numpy()
    trace_jitter = rng.normal(0, 0.055, len(values))
    rate_ax.scatter(
        pos + trace_jitter, values, s=32,
        color=depth_colors[depth_bin], alpha=0.55,
        edgecolor="none", zorder=ZORDER["marker"],
    )
    mouse_med = subset.groupby("subject_id")["approx_spike_rate_hz"].median().dropna().to_numpy()
    mouse_jitter = rng.normal(0, 0.035, len(mouse_med))
    rate_ax.scatter(
        pos + mouse_jitter, mouse_med, s=70,
        facecolor="white", edgecolor=depth_colors[depth_bin], linewidth=1.5,
        zorder=ZORDER["annotation"],
    )
    if len(values):
        med = np.nanmedian(values)
        rate_ax.plot([pos - 0.22, pos + 0.22], [med, med], color=CHARCOAL, lw=2.6)

rate_ax.set_xticks(range(1, len(depth_order) + 1))
rate_ax.set_xticklabels(depth_order, rotation=25, ha="right")
rate_ax.set_ylabel("Approximate spike rate (Hz)")
rate_ax.set_title("Spike-rate distribution by depth")
rate_ax.grid(axis="y", color=LIGHT_GRAY, alpha=0.55)

# Slide-requested burst fraction vs burst frequency.
for depth_bin in depth_order:
    subset = roi_metrics.loc[roi_metrics["depth_bin"] == depth_bin]
    burst_ax.scatter(
        subset["burst_event_rate_hz"],
        100 * subset["spike_burst_fraction"],
        s=54,
        color=depth_colors[depth_bin],
        edgecolor=CHARCOAL,
        linewidth=0.7,
        alpha=0.82,
        label=depth_bin,
    )
burst_ax.set_xlabel("Burst event rate (Hz)")
burst_ax.set_ylabel("Spikes assigned to bursts (%)")
burst_ax.set_title("Burst recruitment across cells")
burst_ax.grid(color=LIGHT_GRAY, alpha=0.55)
burst_ax.legend(title="Depth bin", fontsize=9)

fig.suptitle(
    f"ASAP8 VIP spike detection and dynamics · {', '.join(TARGET_SESSION_TYPES)}",
    color=NAVY, y=0.995,
)
fig.tight_layout()
save_panel(fig, "slide10_spike_detection_rate_bursting")
plt.show()


# Part II — Isolated spike waveform phenotype

The first panel scales to many cells by using a heat map of each ROI's **median baseline-z-scored isolated waveform**, sorted by depth. The second panel shows the depth-bin median and interquartile spread across cells.


In [ ]:
waveform_records = []
for row in roi_metrics.itertuples():
    unit = unit_store[_row_key(row)]
    if unit["n_waveform_spikes"] <= 0:
        continue
    waveform_records.append({
        "key": _row_key(row),
        "subject_id": str(row.subject_id),
        "session_id": str(row.session_id),
        "dmd": int(row.dmd),
        "roi": int(row.roi),
        "depth_um": float(row.depth_um),
        "depth_bin": str(row.depth_bin),
        "width_ms": float(row.median_isolated_width50_ms),
        "n": unit["n_waveform_spikes"],
        "time_ms": unit["waveform_time_ms"],
        "median_z": unit["waveform_median_z"],
    })

waveform_records = sorted(
    waveform_records,
    key=lambda r: (depth_order.index(r["depth_bin"]), r["depth_um"], np.nan_to_num(r["width_ms"], nan=np.inf)),
)

if waveform_records:
    common_time = waveform_records[0]["time_ms"]
    same_grid = all(
        len(r["time_ms"]) == len(common_time)
        and np.allclose(r["time_ms"], common_time, atol=1e-6)
        for r in waveform_records
    )
    if not same_grid:
        # Rare if sample rates differ slightly across sessions: interpolate onto the first grid.
        for r in waveform_records:
            r["median_z"] = np.interp(common_time, r["time_ms"], r["median_z"])
            r["time_ms"] = common_time

    waveform_matrix = np.vstack([r["median_z"] for r in waveform_records])

    fig, axes = plt.subplots(1, 2, figsize=(13.6, 6.7), gridspec_kw={"width_ratios": [1.1, 1.0]})

    finite = waveform_matrix[np.isfinite(waveform_matrix)]
    vmax = np.nanpercentile(finite, 98.5) if len(finite) else 5
    vmin = np.nanpercentile(finite, 2) if len(finite) else -2
    if vmin < 0 < vmax:
        norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    else:
        norm = Normalize(vmin=vmin, vmax=vmax)

    im = axes[0].imshow(
        waveform_matrix,
        aspect="auto",
        interpolation="nearest",
        extent=[common_time[0], common_time[-1], len(waveform_records) - 0.5, -0.5],
        cmap="RdBu_r",
        norm=norm,
    )
    axes[0].axvline(0, color=CHARCOAL, lw=0.8, alpha=0.7)
    axes[0].set_xlabel("Time from detected spike (ms)")
    axes[0].set_ylabel("ROI, sorted by depth")
    axes[0].set_title("Median isolated waveform per ROI\n(pre-spike baseline z-score)")
    fig.colorbar(im, ax=axes[0], label="Baseline SD")

    # Depth-bin separators and labels.
    row_cursor = 0
    for depth_bin in depth_order:
        n_here = sum(r["depth_bin"] == depth_bin for r in waveform_records)
        if n_here == 0:
            continue
        middle = row_cursor + (n_here - 1) / 2
        axes[0].text(
            common_time[-1] + 0.03 * (common_time[-1] - common_time[0]),
            middle,
            depth_bin,
            va="center", ha="left", color=depth_colors[depth_bin], fontsize=9,
            clip_on=False,
        )
        row_cursor += n_here
        if row_cursor < len(waveform_records):
            axes[0].axhline(row_cursor - 0.5, color="white", lw=1.2)

    for depth_bin in depth_order:
        profiles = [r["median_z"] for r in waveform_records if r["depth_bin"] == depth_bin]
        if not profiles:
            continue
        profiles = np.vstack(profiles)
        q25, med, q75 = np.nanpercentile(profiles, [25, 50, 75], axis=0)
        axes[1].fill_between(
            common_time, q25, q75,
            color=depth_colors[depth_bin], alpha=0.20, linewidth=0,
        )
        axes[1].plot(
            common_time, med,
            color=depth_colors[depth_bin], lw=2.3,
            label=f"{depth_bin} (n={len(profiles)})",
        )
    axes[1].axvline(0, color=LIGHT_GRAY, lw=0.9)
    axes[1].axhline(0, color=LIGHT_GRAY, lw=0.9)
    axes[1].set_xlabel("Time from detected spike (ms)")
    axes[1].set_ylabel("Baseline-z-scored dF/F")
    axes[1].set_title("Waveform phenotype by cortical depth")
    axes[1].legend(fontsize=9)
    axes[1].grid(color=LIGHT_GRAY, alpha=0.35)

    fig.suptitle("ASAP8 isolated optical spike waveforms", color=NAVY, y=1.01)
    fig.tight_layout()
    save_panel(fig, "slide11_isolated_waveforms_by_depth")
    plt.show()
else:
    print("No waveform-quality isolated spikes were found.")


## 9. Electrophysiological feature relationships

These are cell-level exploratory plots. The same cells recur across panels; do not interpret each point as an independent animal replicate.


In [ ]:
feature_pairs = [
    ("median_isolated_width50_ms", "median_plateau_index", "Spike width at half-height (ms)", "Median plateau index"),
    ("approx_spike_rate_hz", "spike_burst_fraction", "Approximate spike rate (Hz)", "Fraction of spikes in bursts"),
    ("median_isolated_rise10_90_ms", "median_isolated_decay90_10_ms", "Rise 10–90% (ms)", "Decay 90–10% (ms)"),
    ("median_isolated_snr", "median_isolated_amplitude_dff", "Isolated spike SNR", "Isolated spike amplitude (dF/F)"),
]

fig, axes = plt.subplots(2, 2, figsize=(11.8, 9.0))
for ax, (xmetric, ymetric, xlabel, ylabel) in zip(axes.flat, feature_pairs):
    for depth_bin in depth_order:
        subset = roi_metrics.loc[roi_metrics["depth_bin"] == depth_bin]
        ax.scatter(
            subset[xmetric], subset[ymetric],
            s=48, color=depth_colors[depth_bin],
            edgecolor=CHARCOAL, linewidth=0.65, alpha=0.78,
            label=depth_bin,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(color=LIGHT_GRAY, alpha=0.45)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title="Depth bin", loc="upper center", ncol=max(1, len(depth_order)))
fig.suptitle("ASAP8 somatic voltage phenotype across cells", color=NAVY, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.94])
save_panel(fig, "slide11_ephys_feature_relationships")
plt.show()


# Part III — Burst structure, plateau phenotype, and depth distributions

The next panel mirrors the slide-12 intent: representative compound events from superficial/deep cells plus the spike-width/plateau relationship. Representative cells are chosen near the **median burst fraction** within their depth bin, among cells with detected bursts, rather than by choosing the most dramatic cell.


In [ ]:
rep_burst_rows = []
for depth_bin in example_bins:
    row = representative_row(
        roi_metrics,
        depth_bin,
        metric="spike_burst_fraction",
        require_bursts=True,
    )
    if row is not None:
        rep_burst_rows.append(row)

n_burst_panels = max(1, len(rep_burst_rows))
fig, axes = plt.subplots(1, n_burst_panels + 1, figsize=(5.0 * (n_burst_panels + 1), 5.2))
axes = np.atleast_1d(axes)

for ax, row in zip(axes[:-1], rep_burst_rows):
    unit = unit_store[_row_key(row)]
    examples = unit["burst_examples"]
    time_ms = unit["burst_time_ms"]
    if examples:
        robust_ranges = [
            np.nanpercentile(ex["waveform"], 98) - np.nanpercentile(ex["waveform"], 2)
            for ex in examples
        ]
        spacing = max(float(np.nanmedian(robust_ranges)) * 1.25, 1e-6)
        for level, ex in enumerate(examples):
            waveform = ex["waveform"]
            offset = level * spacing
            ax.plot(time_ms, waveform + offset, color=depth_colors[str(row.depth_bin)], lw=1.2)
            spikes_ms = ex["relative_spikes_ms"]
            spike_values = np.interp(spikes_ms, time_ms, waveform)
            ax.scatter(
                spikes_ms, spike_values + offset,
                marker="v", s=22, facecolor=CHARCOAL,
                edgecolor="white", linewidth=0.35,
            )
            ax.text(
                time_ms[-1], offset,
                f"{ex['n_inferred_spikes']} spikes · {ex['event_window_duration_ms']:.0f} ms",
                ha="right", va="bottom", fontsize=8,
            )
    ax.axvline(0, color=LIGHT_GRAY, lw=0.9)
    ax.set_yticks([])
    ax.set_xlabel("Time from first spike (ms)")
    ax.set_title(
        f"{row.depth_bin}\n{row.subject_id} · DMD{int(row.dmd)} R{int(row.roi)} · {row.depth_um:.0f} µm"
    )

scatter_ax = axes[-1]
max_fraction = max(float(roi_metrics["spike_burst_fraction"].max()), 1e-9)
for depth_bin in depth_order:
    subset = roi_metrics.loc[roi_metrics["depth_bin"] == depth_bin]
    sizes = 35 + 220 * subset["spike_burst_fraction"].fillna(0) / max_fraction
    scatter_ax.scatter(
        subset["median_isolated_width50_ms"],
        subset["median_plateau_index"],
        s=sizes,
        color=depth_colors[depth_bin],
        edgecolor=CHARCOAL,
        linewidth=0.7,
        alpha=0.80,
        label=depth_bin,
    )
scatter_ax.set_xlabel("Isolated spike width at half-height (ms)")
scatter_ax.set_ylabel("Median plateau index")
scatter_ax.set_title("Spike width vs sustained depolarization\n(point size = burst fraction)")
scatter_ax.grid(color=LIGHT_GRAY, alpha=0.45)
scatter_ax.legend(title="Depth bin", fontsize=9)

fig.suptitle("ASAP8 burst structure and plateau phenotype", color=NAVY, y=1.01)
fig.tight_layout()
save_panel(fig, "slide12_burst_structure_and_plateau")
plt.show()


In [ ]:
plot_metrics = [
    ("approx_spike_rate_hz", "Approximate spike rate (Hz)"),
    ("median_isolated_width50_ms", "Isolated spike width at half-height (ms)"),
    ("spike_burst_fraction", "Fraction of detected spikes in bursts"),
    ("median_burst_duration_ms", "Median burst-window duration (ms)"),
    ("median_plateau_index", "Median plateau index"),
    ("median_isolated_snr", "Isolated spike SNR"),
]

fig, axes = plt.subplots(2, 3, figsize=(14.0, 8.2))
rng = np.random.default_rng(4)

for ax, (metric, ylabel) in zip(axes.flat, plot_metrics):
    for pos, depth_bin in enumerate(depth_order, start=1):
        subset = roi_metrics.loc[roi_metrics["depth_bin"] == depth_bin]
        values = subset[metric].dropna().to_numpy()
        if len(values):
            ax.scatter(
                pos + rng.normal(0, 0.055, len(values)), values,
                s=30, color=depth_colors[depth_bin], alpha=0.45, edgecolor="none",
            )
            ax.plot(
                [pos - 0.20, pos + 0.20],
                [np.nanmedian(values)] * 2,
                color=CHARCOAL, lw=2.4,
            )

        # One open point per mouse prevents the cell-rich sessions from hiding the hierarchy.
        mouse_med = subset.groupby("subject_id")[metric].median().dropna().to_numpy()
        if len(mouse_med):
            ax.scatter(
                pos + rng.normal(0, 0.032, len(mouse_med)), mouse_med,
                s=62, facecolor="white", edgecolor=depth_colors[depth_bin],
                linewidth=1.4, zorder=ZORDER["annotation"],
            )

    ax.set_xticks(range(1, len(depth_order) + 1))
    ax.set_xticklabels(depth_order, rotation=28, ha="right")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", color=LIGHT_GRAY, alpha=0.5)
    ax.margins(x=0.15)

fig.suptitle(
    "ASAP8 somatic voltage phenotype by 100-µm cortical depth bin\n"
    "filled = cells; open = mouse medians; bar = cell median",
    color=NAVY, y=1.02,
)
fig.tight_layout()
save_panel(fig, "slide12_depth_binned_phenotype_summary")
plt.show()


# Part IV — ROI correlations and spike synchrony

Every pair below is same-mouse/same-session by construction. Cross-DMD pairs are allowed **only when both DMDs belong to the same session**, because the biological question is synchrony across simultaneously recorded depths, not similarity between unrelated recordings.

For pooled comparisons, the primary plotted replicate is a **session median within each depth-bin pair**. Raw pair values remain in `pair_df` for inspection and modeling.


In [ ]:
if pair_df.empty:
    print("No within-session pairs were available.")
else:
    # Explicit audit: pair construction cannot cross sessions, and there should be no duplicate session ambiguity.
    pair_audit = (
        pair_df.groupby(["subject_id", "session_id"], observed=True)
        .size()
        .rename("n_pairs")
        .reset_index()
    )
    display(pair_audit)

    pair_summary_metrics = [
        "spike_sttc_5ms",
        "isolated_spike_sttc_5ms",
        "burst_onset_sttc_20ms",
        "spike_count_corr_20ms",
        "spike_count_corr_250ms",
        "burst_state_jaccard",
    ]
    session_pair_summary = (
        pair_df.groupby(
            [
                "subject_id", "session_id", "session_type",
                "depth_bin_pair", "depth_bin_pair_start_a", "depth_bin_pair_start_b",
            ],
            observed=True,
        )[pair_summary_metrics]
        .median()
        .reset_index()
    )
    session_pair_summary["n_raw_pairs"] = (
        pair_df.groupby(
            [
                "subject_id", "session_id", "session_type",
                "depth_bin_pair", "depth_bin_pair_start_a", "depth_bin_pair_start_b",
            ],
            observed=True,
        ).size().to_numpy()
    )

    pair_order_df = (
        session_pair_summary[["depth_bin_pair", "depth_bin_pair_start_a", "depth_bin_pair_start_b"]]
        .drop_duplicates()
        .sort_values(["depth_bin_pair_start_a", "depth_bin_pair_start_b"])
    )
    pair_order = pair_order_df["depth_bin_pair"].tolist()
    display(session_pair_summary.head())


## 13. Example-session raster and synchrony matrices

The example session is selected objectively as the successfully processed session with the most cells. The matrices include within-DMD and cross-DMD pairs, but all are simultaneous within that asset.


In [ ]:
if not pair_df.empty:
    exemplar = session_summary_df.sort_values(["n_cells", "n_pairs"], ascending=False).iloc[0]
    ex_subject = str(exemplar.subject_id)
    ex_session = str(exemplar.session_id)
    ex_units = [
        unit for key, unit in unit_store.items()
        if key[0] == ex_subject and key[1] == ex_session
    ]
    ex_units = sorted(ex_units, key=lambda u: (u["depth_bin_start_um"], u["depth_um"], u["dmd"], u["roi"]))

    labels = [f"{u['depth_um']:.0f}µm D{u['dmd']}R{u['roi']}" for u in ex_units]
    key_to_index = {
        (u["dmd"], u["roi"]): i for i, u in enumerate(ex_units)
    }

    sttc_matrix = np.eye(len(ex_units))
    burst_matrix = np.eye(len(ex_units))
    ex_pairs = pair_df.loc[
        (pair_df["subject_id"].astype(str) == ex_subject)
        & (pair_df["session_id"].astype(str) == ex_session)
    ]
    for row in ex_pairs.itertuples():
        i = key_to_index[(row.unit_a_dmd, row.unit_a_roi)]
        j = key_to_index[(row.unit_b_dmd, row.unit_b_roi)]
        sttc_matrix[i, j] = sttc_matrix[j, i] = row.spike_sttc_5ms
        burst_matrix[i, j] = burst_matrix[j, i] = row.burst_state_jaccard

    fig, axes = plt.subplots(
        1, 3, figsize=(16.0, 5.4),
        gridspec_kw={"width_ratios": [1.35, 1.0, 1.0]},
    )
    r0, r1 = SYNCH_RASTER_WINDOW_SEC
    for row_idx, unit in enumerate(ex_units):
        spikes = unit["spike_times_sec"]
        spikes = spikes[(spikes >= r0) & (spikes <= r1)]
        axes[0].vlines(
            spikes, row_idx - 0.34, row_idx + 0.34,
            color=depth_colors[unit["depth_bin"]], lw=0.95,
        )
        burst_times = unit["burst_onset_times_sec"]
        burst_times = burst_times[(burst_times >= r0) & (burst_times <= r1)]
        axes[0].scatter(
            burst_times, np.full(len(burst_times), row_idx),
            s=24, facecolor="white", edgecolor=depth_colors[unit["depth_bin"]],
            linewidth=1.0,
        )
    axes[0].set_yticks(np.arange(len(labels)))
    axes[0].set_yticklabels(labels)
    axes[0].invert_yaxis()
    axes[0].set_xlim(r0, r1)
    axes[0].set_xlabel("Time (s)")
    axes[0].set_title("Detected spikes and burst onsets")

    offdiag_sttc = sttc_matrix[np.eye(len(ex_units), dtype=bool) == 0]
    sttc_vmax = max(0.25, np.nanpercentile(offdiag_sttc, 98)) if len(offdiag_sttc) else 1
    im1 = axes[1].imshow(sttc_matrix, cmap=SYNC_CMAP, vmin=0, vmax=sttc_vmax)
    axes[1].set_title("Spike STTC · ±5 ms")

    offdiag_burst = burst_matrix[np.eye(len(ex_units), dtype=bool) == 0]
    burst_vmax = max(0.25, np.nanpercentile(offdiag_burst, 98)) if len(offdiag_burst) else 1
    im2 = axes[2].imshow(burst_matrix, cmap=SYNC_CMAP, vmin=0, vmax=burst_vmax)
    axes[2].set_title("Burst-state temporal overlap")

    for ax in axes[1:]:
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
        ax.set_yticklabels(labels, fontsize=7)

    fig.colorbar(im1, ax=axes[1], label="STTC", fraction=0.046, pad=0.04)
    fig.colorbar(im2, ax=axes[2], label="Jaccard overlap", fraction=0.046, pad=0.04)
    fig.suptitle(
        f"Within-session synchrony · subject {ex_subject} · session {ex_session}",
        color=NAVY, y=1.01,
    )
    fig.tight_layout()
    save_panel(fig, "slide13_example_session_synchrony")
    plt.show()


## 14. Synchrony across cortical depth pairs

Each dot is one **session median** for the corresponding pair of 100-µm depth bins. This avoids giving a session with many cells—and therefore many combinatorial pairs—disproportionate inferential weight.


In [ ]:
if not pair_df.empty:
    sync_metrics = [
        ("spike_sttc_5ms", "Spike STTC · ±5 ms"),
        ("burst_onset_sttc_20ms", "Burst-onset STTC · ±20 ms"),
        ("spike_count_corr_250ms", "Spike-count correlation · 250 ms"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15.2, 5.0), sharex=True)
    rng = np.random.default_rng(11)

    # Color each pair by mean depth of its two bins.
    pair_colors = {}
    for row in pair_order_df.itertuples():
        mean_start = 0.5 * (row.depth_bin_pair_start_a + row.depth_bin_pair_start_b)
        pair_colors[row.depth_bin_pair] = DEPTH_CMAP(depth_norm(mean_start + DEPTH_BIN_UM / 2))

    for ax, (metric, title) in zip(axes, sync_metrics):
        for pos, pair_label in enumerate(pair_order, start=1):
            values = session_pair_summary.loc[
                session_pair_summary["depth_bin_pair"] == pair_label, metric
            ].dropna().to_numpy()
            if len(values):
                ax.scatter(
                    pos + rng.normal(0, 0.05, len(values)), values,
                    s=60, color=pair_colors[pair_label],
                    edgecolor=CHARCOAL, linewidth=0.8, alpha=0.82,
                )
                ax.plot(
                    [pos - 0.19, pos + 0.19], [np.nanmedian(values)] * 2,
                    color=CHARCOAL, lw=2.5,
                )
        ax.axhline(0, color=LIGHT_GRAY, lw=0.9)
        ax.grid(axis="y", color=LIGHT_GRAY, alpha=0.50)
        ax.set_xticks(range(1, len(pair_order) + 1))
        ax.set_xticklabels(pair_order, rotation=55, ha="right", fontsize=8)
        ax.set_title(title)

    fig.suptitle(
        "Pairwise synchrony across depth · one summary value per session × depth-bin pair",
        color=NAVY, y=1.02,
    )
    fig.tight_layout()
    save_panel(fig, "slide14_synchrony_by_depth_pair")
    plt.show()


## 15. Representative cross-correlograms

For each depth-bin pair, the displayed pair is the raw pair whose 5-ms STTC is **closest to that depth-pair's median**, rather than the strongest pair. This is meant to show a representative cross-correlation shape without selecting an extreme.


In [ ]:
if not pair_df.empty:
    chosen_pairs = []
    for pair_label in pair_order:
        subset = pair_df.loc[
            (pair_df["depth_bin_pair"] == pair_label)
            & np.isfinite(pair_df["spike_sttc_5ms"])
        ].copy()
        if subset.empty:
            continue
        target = subset["spike_sttc_5ms"].median()
        chosen_pairs.append(
            subset.iloc[np.argmin(np.abs(subset["spike_sttc_5ms"].to_numpy() - target))]
        )

    chosen_pairs = chosen_pairs[:MAX_CCG_PANELS]
    if chosen_pairs:
        ncols = min(3, len(chosen_pairs))
        nrows = int(np.ceil(len(chosen_pairs) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.0 * nrows), squeeze=False)

        for ax, row in zip(axes.flat, chosen_pairs):
            key_a = unit_key(row.subject_id, row.session_id, row.unit_a_dmd, row.unit_a_roi)
            key_b = unit_key(row.subject_id, row.session_id, row.unit_b_dmd, row.unit_b_roi)
            unit_a = unit_store[key_a]
            unit_b = unit_store[key_b]

            lag, conditional_rate = cross_correlogram(
                unit_a["spike_times_sec"], unit_b["spike_times_sec"],
                window_s=0.050, bin_s=0.001,
            )
            baseline_rate = len(unit_b["spike_times_sec"]) / unit_b["duration_s"]
            excess = conditional_rate - baseline_rate

            mean_start = 0.5 * (row.depth_bin_pair_start_a + row.depth_bin_pair_start_b)
            color = DEPTH_CMAP(depth_norm(mean_start + DEPTH_BIN_UM / 2))
            ax.fill_between(lag * 1000, excess, 0, color=color, alpha=0.22, linewidth=0)
            ax.plot(lag * 1000, excess, color=color, lw=2.0)
            ax.axvline(0, color=LIGHT_GRAY, lw=0.9)
            ax.axhline(0, color=LIGHT_GRAY, lw=0.9)
            ax.set_xlabel("Lag (ms)")
            ax.set_ylabel("Excess conditional spike rate (Hz)")
            ax.set_title(
                f"{row.depth_bin_pair}\n"
                f"{row.subject_id} · {row.session_id} · STTC={row.spike_sttc_5ms:.2f}"
            )

        for ax in axes.flat[len(chosen_pairs):]:
            ax.axis("off")

        fig.suptitle("Representative within-session spike cross-correlograms", color=NAVY, y=1.01)
        fig.tight_layout()
        save_panel(fig, "slide14_representative_cross_correlograms")
        plt.show()


## 16. Synchrony versus physical depth separation

This is a useful secondary diagnostic for deciding whether discrete 100-µm bins are hiding a continuous depth relationship. Raw pairs are shown faintly; the biological replicate remains the session.


In [ ]:
if not pair_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.8))
    axes[0].scatter(
        pair_df["depth_separation_um"], pair_df["spike_sttc_5ms"],
        s=22, color=GRAY, alpha=0.22, edgecolor="none",
    )
    axes[0].set_xlabel("Absolute depth separation (µm)")
    axes[0].set_ylabel("Spike STTC · ±5 ms")
    axes[0].set_title("Fast synchrony vs depth separation")

    axes[1].scatter(
        pair_df["depth_separation_um"], pair_df["spike_count_corr_250ms"],
        s=22, color=GRAY, alpha=0.22, edgecolor="none",
    )
    axes[1].set_xlabel("Absolute depth separation (µm)")
    axes[1].set_ylabel("Spike-count correlation · 250 ms")
    axes[1].set_title("Slower co-modulation vs depth separation")

    for ax in axes:
        ax.axhline(0, color=LIGHT_GRAY, lw=0.9)
        ax.grid(color=LIGHT_GRAY, alpha=0.35)

    fig.tight_layout()
    save_panel(fig, "supp_synchrony_vs_depth_separation")
    plt.show()


# Part V — Save cohort outputs

The cohort-level tables are intentionally compact. If `SAVE_SESSION_EVENT_TABLES=True`, every session also gets its own full spike/event tables under `tables/sessions/<session_id>/`.


In [ ]:
if SAVE_TABLES:
    roi_metrics.to_csv(TABLE_DIR / "asap8_roi_ephys_metrics_cohort.csv", index=False)
    session_summary_df.to_csv(TABLE_DIR / "session_processing_summary.csv", index=False)
    failures_df.to_csv(TABLE_DIR / "session_processing_failures.csv", index=False)

    if not pair_df.empty:
        pair_df.to_csv(TABLE_DIR / "asap8_pairwise_synchrony_within_session.csv", index=False)
        session_pair_summary.to_csv(
            TABLE_DIR / "asap8_pairwise_synchrony_session_medians.csv", index=False
        )

    # Save compact per-ROI waveform profiles with a metadata table.
    waveform_arrays = {}
    waveform_meta = []
    for key, unit in unit_store.items():
        safe = re.sub(r"[^A-Za-z0-9_-]+", "_", f"{key[0]}_{key[1]}_D{key[2]}_R{key[3]}")
        waveform_arrays[f"{safe}__time_ms"] = unit["waveform_time_ms"]
        waveform_arrays[f"{safe}__p10_z"] = unit["waveform_p10_z"]
        waveform_arrays[f"{safe}__median_z"] = unit["waveform_median_z"]
        waveform_arrays[f"{safe}__p90_z"] = unit["waveform_p90_z"]
        waveform_meta.append({
            "array_prefix": safe,
            "subject_id": unit["subject_id"],
            "session_id": unit["session_id"],
            "session_type": unit["session_type"],
            "dmd": unit["dmd"],
            "roi": unit["roi"],
            "depth_um": unit["depth_um"],
            "depth_bin": unit["depth_bin"],
            "n_waveform_spikes": unit["n_waveform_spikes"],
        })
    np.savez_compressed(TABLE_DIR / "asap8_isolated_waveform_profiles_z.npz", **waveform_arrays)
    pd.DataFrame(waveform_meta).to_csv(TABLE_DIR / "asap8_isolated_waveform_profiles_metadata.csv", index=False)

print("Saved cohort outputs to:", OUTPUT_ROOT)


## 17. Recommended inferential next step

For a lab-meeting descriptive figure, the cell distributions plus mouse/session summaries above are appropriate. If a depth effect becomes a formal result, avoid a simple cell-level t-test/ANOVA.

A good next analysis is a hierarchical model or hierarchical bootstrap:

- **cell metrics:** depth bin as the fixed effect, mouse (and session if multiple matching sessions per mouse) as the grouping structure;
- **pair metrics:** session-level summaries as the primary replicate, or a model that explicitly accounts for repeated participation of the same neurons in multiple pairs;
- **continuous depth check:** repeat the analysis using exact `depth_um` rather than bins to verify that a bin boundary is not driving the effect.

The 100-µm bins are therefore a presentation/stratification device, not a claim that physiology changes discretely at 100-µm boundaries.
